# Game 3 account-filter impact

Compare the original and corrected feature eligibility on the same registered-player cohort. Hold the chronology, binary target, eight model inputs, model settings and group membership fixed. Fit each version separately and use only the historical validation players to calibrate scores and choose a threshold. Differences in ROC-AUC use paired, outcome-stratified bootstrap intervals conditional on the fitted models.

In [1]:
import os
import sys
from pathlib import Path

candidates = (
    [Path(os.environ["CHURN_PROJECT_ROOT"])] if "CHURN_PROJECT_ROOT" in os.environ else
    [Path.cwd() / ".pipeline", Path.cwd(), Path.cwd().parent / ".pipeline", Path.cwd().parent]
)
ROOT = next((candidate.resolve() for candidate in candidates
             if (candidate / "src" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run 00_setup.ipynb first or point CHURN_PROJECT_ROOT at the pipeline checkout.")
sys.path.insert(0, str(ROOT))



In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src import config, features, labeling

INPUTS = ["recency_hours", "play_count", "active_days", "active_duration_hours",
          "session_count", "weekend_ratio", "night_ratio", "consecutive_play_ratio"]
BOOTSTRAPS = 2000
SEED = 42

def temporal_split(table):
    idx = table.first_ts
    test_start = idx.iloc[int(.8 * len(table))]
    first_test_prediction = test_start + config.OBSERVATION_PERIOD_DAYS * labeling.DAY
    earlier = table[(idx < test_start) & (table.cp_end <= first_test_prediction)]
    val_start = earlier.first_ts.iloc[int(.75 * len(earlier))]
    train = earlier[earlier.first_ts < val_start]
    val = earlier[earlier.first_ts >= val_start]
    test = table[idx >= test_start]
    assert earlier.cp_end.max() <= first_test_prediction
    assert train.first_ts.max() < val.first_ts.min() < test.first_ts.min()
    assert len(set(train.player_id) | set(val.player_id) | set(test.player_id)) == sum(
        len(part) for part in (train, val, test))
    return train, val, test

def fit(train, val, test, columns):
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(max_iter=2000, class_weight="balanced"))
    model.fit(train[columns], train.churn)
    val_raw = model.predict_proba(val[columns])[:, 1]
    test_raw = model.predict_proba(test[columns])[:, 1]
    calibrator = LogisticRegression(C=1000, max_iter=1000).fit(val_raw[:, None], val.churn)
    validation_score = calibrator.predict_proba(val_raw[:, None])[:, 1]
    test_score = calibrator.predict_proba(test_raw[:, None])[:, 1]
    fpr, tpr, thresholds = roc_curve(val.churn, validation_score, drop_intermediate=False)
    threshold = thresholds[1:][np.argmax((tpr-fpr)[1:])]
    return test_score, float(threshold)

def score(y, p, threshold):
    retained = y == 0
    return {"n": len(y), "retained": int(retained.sum()),
            "retained_correct": int((retained & (p < threshold)).sum()),
            "roc_auc": float(roc_auc_score(y, p)),
            "balanced_accuracy": float(balanced_accuracy_score(y, p >= threshold)),
            "brier": float(brier_score_loss(y, p)),
            "average_precision_retained": float(average_precision_score(1-y, 1-p)),
            "threshold": threshold}

def auc_interval(y1, p1, y2, p2, paired=False):
    rng = np.random.default_rng(SEED if paired else SEED + 1)
    cells1 = [np.flatnonzero(y1 == value) for value in (0, 1)]
    cells2 = [np.flatnonzero(y2 == value) for value in (0, 1)]
    assert all(len(cell) for cell in cells1 + cells2)

    def sample(cells):
        return np.concatenate([rng.choice(cell, len(cell), replace=True) for cell in cells])

    def auc_ordered(scores, group):
        neg, pos = map(len, group)
        return (rankdata(scores)[neg:].sum() - pos * (pos + 1) / 2) / (pos * neg)

    changes = []
    for _ in range(BOOTSTRAPS):
        i = sample(cells1)
        j = i if paired else sample(cells2)
        changes.append(auc_ordered(p1[i], cells1) - auc_ordered(p2[j], cells2))
    return np.quantile(changes, [.025, .975]).tolist()

In [3]:
def prepare_features():
    events = pd.read_parquet(config.INTERIM_DIR / "game3_events.parquet",
                             columns=["player_id", "ts", "authed"])
    cohort = labeling.build_cohort(events)
    registered = events.authed.fillna(True)
    # The common inputs use timing and counts, so game-specific fields are unused.
    events = events.assign(**dict.fromkeys(
        ("score", "won", "action", "fail_state", "duration", "price", "platform", "device"), np.nan
    ))

    def build(rows):
        matches = features.build_features(rows, cohort, config.OBSERVATION_PERIOD_DAYS)
        return cohort[["player_id", "first_ts", "churn", "cp_end", "n_events_op"]].merge(
            matches[["player_id", *INPUTS]], on="player_id", validate="1:1")

    legacy = build(events)
    corrected = build(events.loc[registered])
    assert len(legacy) == len(corrected) == 24128
    assert legacy.player_id.equals(corrected.player_id)
    assert np.array_equal(corrected.play_count, corrected.n_events_op)
    assert (legacy.play_count >= corrected.play_count).all()
    return corrected.sort_values("first_ts", kind="stable"), legacy.set_index("player_id")

def evaluate():
    corrected, legacy = prepare_features()
    train, validation, test = temporal_split(corrected)
    legacy_splits = [legacy.loc[part.player_id] for part in (train, validation, test)]
    y = test.churn.to_numpy()
    assert all(np.array_equal(old.churn, new.churn)
               for old, new in zip(legacy_splits, (train, validation, test)))
    old_score, old_threshold = fit(*legacy_splits, INPUTS)
    new_score, new_threshold = fit(train, validation, test, INPUTS)
    light_max = int(train.play_count.quantile(.25, interpolation="higher"))
    heavy_min = max(light_max + 1, int(train.play_count.quantile(.75, interpolation="higher")))
    activity = test.play_count.to_numpy()
    comparison = {}
    for name, mask in {
        "all": np.ones(len(test), dtype=bool),
        "light": activity <= light_max,
        "heavy": activity >= heavy_min,
    }.items():
        before = score(y[mask], old_score[mask], old_threshold)
        after = score(y[mask], new_score[mask], new_threshold)
        comparison[name] = {
            "legacy": before,
            "corrected": after,
            "auc_delta_interval": auc_interval(
                y[mask], new_score[mask], y[mask], old_score[mask], paired=True),
            "auc_delta": after["roc_auc"] - before["roc_auc"],
        }
    return {
        "cohort_players": len(corrected),
        "count_mismatches": int((legacy.loc[corrected.player_id].play_count.to_numpy()
                                  != corrected.play_count.to_numpy()).sum()),
        "test_count_mismatches": int((legacy_splits[2].play_count.to_numpy()
                                       != test.play_count.to_numpy()).sum()),
        "split_counts": [len(train), len(validation), len(test)],
        "light_events_at_most": light_max,
        "heavy_events_at_least": heavy_min,
        "comparison": comparison,
        "bootstrap_replicates": BOOTSTRAPS,
        "bootstrap_seed": SEED,
    }

In [4]:
import json
from importlib.metadata import version
results = evaluate()
results["packages"] = {name: version(name) for name in ("numpy", "pandas", "scipy", "scikit-learn", "pyarrow")}
assert results["count_mismatches"] == 2371
assert results["split_counts"] == [13627, 4543, 4826]
(ROOT / "game3_filter_impact_results.json").write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")
print(json.dumps(results, indent=2))

  cohort: 26,868 players -> 25,705 after left guard -> 24,128 after right guard -> 24,128 with OP activity | churn rate 60.0%


{
  "cohort_players": 24128,
  "count_mismatches": 2371,
  "test_count_mismatches": 668,
  "split_counts": [
    13627,
    4543,
    4826
  ],
  "light_events_at_most": 1,
  "heavy_events_at_least": 8,
  "comparison": {
    "all": {
      "legacy": {
        "n": 4826,
        "retained": 1616,
        "retained_correct": 1180,
        "roc_auc": 0.802572888714105,
        "balanced_accuracy": 0.7618279818636069,
        "brier": 0.15299167621302923,
        "average_precision_retained": 0.7215820170990749,
        "threshold": 0.6876016577760585
      },
      "corrected": {
        "n": 4826,
        "retained": 1616,
        "retained_correct": 1181,
        "roc_auc": 0.8114252722001172,
        "balanced_accuracy": 0.7663429952808365,
        "brier": 0.1494277946706094,
        "average_precision_retained": 0.7393039007281825,
        "threshold": 0.6741059718751267
      },
      "auc_delta_interval": [
        0.003530198887295266,
        0.014302236590481427
      ],
      "